# BYOD — EPMA template (`petropandas`)

**Pour in your own microprobe export, run all cells, collect the results in `output/`.**

1. Edit **only** the `CONFIG` cell below.
2. *Run → Run All Cells*.
3. Check the QC summary, then look at `output/<SAMPLE>_*.xlsx|pdf`.

As shipped, the config points to the workshop example file, so the notebook runs as-is.

In [ ]:
from pathlib import Path
from petropandas import pd, mdb, TernaryPlot, ScatterPlot, ProfilePlot
from petropandas import Grt, Fsp, Bt, Ms, Amp, Cpx, Opx, Chl, St, Ep, Cld, Crd, Ilm, Spl, Ttn, GrtFe3

## CONFIG — edit this cell

In [ ]:
SAMPLE = "S12"                         # used in output file names
FILE = Path("data/epma_export.xlsx")   # .xlsx, .xls or .csv
SHEET = "Spots"                        # Excel sheet name (ignored for .csv)
SKIPROWS = 0                           # header lines above the column names in the export
INDEX_COL = "Comment"                  # column used as row labels (or None)

# Rename odd column names to what petropandas understands, e.g. {"SiO2(Mass%)": "SiO2"}.
# Common variants like "FeO*", "FeOT", "FeO tot", "H2O+" are recognised automatically.
RENAME = {}

# How to find the mineral of each analysis:
#   - a column that already holds it:            MINERAL_COLUMN = "Mineral"
#   - or a regex that extracts it from a column:  MINERAL_FROM = ("Comment", r"_([a-z]+)_")
MINERAL_COLUMN = None
MINERAL_FROM = ("Comment", r"_([a-z]+)_")

# label found in the data -> (mineral definition, acceptable oxide-total window)
MINERALS = {
    "grt": (Grt, (98.5, 101.5)),
    "pl":  (Fsp, (98.5, 101.5)),
    "bt":  (Bt,  (94.0, 97.5)),
    "ms":  (Ms,  (94.0, 97.0)),
}
MIN_QUALITY = 0.95                     # stoichiometry_quality threshold (0-1)

# Optional traverse sheet: (sheet name, distance column) or None
PROFILE = ("Grt-profile", "Distance_um")
PROFILE_MINERAL = Grt

## 1. Load

In [ ]:
OUT = Path("output"); OUT.mkdir(exist_ok=True)

if FILE.suffix.lower() == ".csv":
    raw = pd.read_csv(FILE, skiprows=SKIPROWS)
else:
    raw = pd.read_excel(FILE, sheet_name=SHEET, skiprows=SKIPROWS)
raw = raw.rename(columns=RENAME)
if INDEX_COL:
    raw = raw.set_index(INDEX_COL, drop=False)

if MINERAL_COLUMN:
    raw["_mineral"] = raw[MINERAL_COLUMN].astype(str)
else:
    col, pattern = MINERAL_FROM
    raw["_mineral"] = raw[col].astype(str).str.extract(pattern, expand=False)

print(f"{len(raw)} analyses, oxide columns found: {list(raw.oxides().columns)}")
print("Unrecognised columns (ignored):", [c for c in raw.columns if c not in raw.oxides().columns])
raw["_mineral"].value_counts(dropna=False)

## 2. Recalculate and quality-check every mineral

In [ ]:
results, summary = {}, []
for label, (mineral, window) in MINERALS.items():
    data = raw.oxides.select(raw["_mineral"] == label)
    if data.empty:
        print(f"no analyses labelled {label!r} - skipped")
        continue
    total = data.oxides().sum(axis=1)
    quality = data.mineral.stoichiometry_quality(mineral)
    ok = total.between(*window) & (quality.fillna(1) >= MIN_QUALITY)
    good = data.oxides.select(ok)
    results[label] = {
        "mineral": mineral,
        "qc": pd.concat([total.rename("Total"), quality, ok.rename("accepted"),
                         data.mineral.check_stoichiometry(mineral)], axis=1),
        "apfu": good.mineral.apfu(mineral),
        "end_members": good.mineral.end_members(mineral),
    }
    summary.append({"label": label, "mineral": mineral.name, "n": len(data), "accepted": int(ok.sum()),
                    "rejected": ", ".join(map(str, data.index[~ok]))})
pd.DataFrame(summary)

Mean composition (± 1σ) of accepted analyses:

In [ ]:
for label, r in results.items():
    em = r["end_members"]
    print(f"\n{r['mineral'].name}  (n = {len(em)})")
    print(em.agg(["mean", "std"]).round(2).to_string())

## 3. Charts

In [ ]:
TERNARIES = {  # mineral -> (top, left, right) end-members
    "Garnet": ("Prp", "Alm", "Grs"),
    "Feldspar": ("An", "Ab", "Or"),
}
for label, r in results.items():
    axes = TERNARIES.get(r["mineral"].name)
    if axes:
        t = TernaryPlot(*axes, title=f"{SAMPLE} {r['mineral'].name}")
        t.add(r["end_members"], label=label)
        t.savefig(OUT / f"{SAMPLE}_{label}_ternary.pdf")
        t.show()

In [ ]:
for label, r in results.items():
    a = r["apfu"]
    if {"Mg{2+}", "Fe{2+}", "Si{4+}"} <= set(a.columns):
        s = ScatterPlot("Si{4+}", "Mg{2+} / (Mg{2+} + Fe{2+})", xlabel="Si (apfu)", ylabel="XMg",
                        title=f"{SAMPLE} {r['mineral'].name}")
        s.add(a, label=label)
        s.savefig(OUT / f"{SAMPLE}_{label}_Si_XMg.pdf")
        s.show()

In [ ]:
if PROFILE:
    sheet, dist = PROFILE
    prof = pd.read_excel(FILE, sheet_name=sheet, skiprows=SKIPROWS).rename(columns=RENAME).set_index(dist)
    prof_em = prof.mineral.end_members(PROFILE_MINERAL)
    p = ProfilePlot(split="auto", xlabel=dist, title=f"{SAMPLE} {PROFILE_MINERAL.name} profile")
    p.add(prof_em, lw=2)
    p.savefig(OUT / f"{SAMPLE}_profile.pdf")
    p.show()
    results["profile"] = {"end_members": prof_em, "apfu": prof.mineral.apfu(PROFILE_MINERAL)}

## 4. Export everything to one workbook

In [ ]:
xlsx = OUT / f"{SAMPLE}_petropandas.xlsx"
with pd.ExcelWriter(xlsx) as w:
    pd.DataFrame(summary).to_excel(w, sheet_name="summary", index=False)
    for label, r in results.items():
        for key in ("qc", "apfu", "end_members"):
            if key in r:
                r[key].round(4).to_excel(w, sheet_name=f"{label}_{key}"[:31])
print("written", xlsx)